# Settimana 18 — Langfuse: dashboard visiva e versionamento dei prompt

**Obiettivo:** collegare ResearchAgent a Langfuse per avere trace visive,
confrontare versioni di prompt e aggiungere punteggi di qualità.

**Al termine avrai:**
- ResearchAgent che invia trace a Langfuse a ogni esecuzione
- Il prompt di sintesi gestito dal pannello di Langfuse
- Punteggi di qualità collegati a ogni trace
- Un `LangfuseTracer` che convive con il `Tracer` proprio della settimana 17

**Prerequisito di piattaforma (prima di eseguire):**
1. Crea un account gratuito su https://cloud.langfuse.com
   (oppure avvia self-hosted: `docker compose up` con il docker-compose di Langfuse)
2. Vai su Settings → API Keys → Create new keys
3. Aggiungi al `.env` di `proyectos/research-agent/`:
   ```
   LANGFUSE_PUBLIC_KEY=pk-lf-...
   LANGFUSE_SECRET_KEY=sk-lf-...
   LANGFUSE_HOST=https://cloud.langfuse.com
   ```

**Se non vuoi creare un account ora:** il notebook ha una modalità `LANGFUSE_ENABLED=False`
che esegue tutto in locale usando solo il `Tracer` della settimana 17.

**Costo stimato:** $0.02–0.05 (2-3 esecuzioni dell'agente).

## Cella 0 — Setup e rilevamento delle credenziali

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "langfuse>=2.0.0", "python-dotenv"])

import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
load_dotenv(PROJECT_DIR / ".env")

sys.path.insert(0, str(PROJECT_DIR))

assert os.getenv("OPENAI_API_KEY"), "Definisci OPENAI_API_KEY nel .env"

# Langfuse è opzionale — il notebook funziona anche senza credenziali
LANGFUSE_PUBLIC = os.getenv("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET = os.getenv("LANGFUSE_SECRET_KEY", "")
LANGFUSE_HOST   = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
LANGFUSE_ENABLED = bool(LANGFUSE_PUBLIC and LANGFUSE_SECRET)

if LANGFUSE_ENABLED:
    print("✓ Credenziali Langfuse trovate — modalità connessa")
    print(f"  Host: {LANGFUSE_HOST}")
else:
    print("⚠  Nessuna credenziale Langfuse — modalità locale (Tracer proprio)")
    print("   Per connetterti: aggiungi LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY al .env")

## Cella 1 — Verificare la connessione con Langfuse

In [ ]:
if LANGFUSE_ENABLED:
    from langfuse import Langfuse

    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC,
        secret_key=LANGFUSE_SECRET,
        host=LANGFUSE_HOST,
    )

    ok = langfuse.auth_check()
    if ok:
        print("✓ Connessione con Langfuse verificata")
        print(f"  Pannello: {LANGFUSE_HOST}")
    else:
        print("✗ Errore di autenticazione — verifica le credenziali")
        LANGFUSE_ENABLED = False
else:
    langfuse = None
    print("Verifica saltata (modalità locale)")

## Cella 2 — Trace manuale: il modello concettuale

Prima di integrare con l'agente, impara il modello Trace → Span → Generation
inviando una trace semplice a mano.

In [ ]:
import time

if LANGFUSE_ENABLED:
    # ── Crea una trace completa a mano ────────────────────────────────────────
    trace = langfuse.trace(
        name="demo_manual",
        input={"tema": "Test di connessione"},
        metadata={"entorno": "notebook", "semana": 18},
    )

    # Span per un'operazione generica (ricerca web)
    span_busqueda = trace.span(
        name="buscar_web",
        input={"query": "IA nella pesca in Galizia"},
    )
    time.sleep(0.3)   # simula latenza
    span_busqueda.end(
        output={"n_resultados": 5},
        metadata={"fuente": "duckduckgo"},
    )

    # Generation per una chiamata all'LLM
    gen = trace.generation(
        name="planificar",
        model="gpt-4o-mini",
        input=[{"role": "user", "content": "Pianifica la ricerca sull'IA nella pesca"}],
        output="1. Cercare applicazioni nell'acquacoltura\n2. Analizzare progetti europei",
        usage={"input": 450, "output": 180},
    )
    gen.end()

    # Chiude la trace con il risultato finale
    trace.update(
        output={"n_subtareas": 2},
        tags=["demo", "semana-18"],
    )

    # IMPORTANTE: flush invia i dati prima che il processo termini
    langfuse.flush()

    print("✓ Trace inviata a Langfuse")
    print(f"  Cercala in: {LANGFUSE_HOST}/traces")
    print(f"  ID della trace: {trace.id}")

else:
    # Modalità locale: mostra la struttura senza inviare
    print("[Modalità locale] Struttura di una trace Langfuse:")
    estructura = {
        "trace": {
            "name": "investigar",
            "input": {"tema": "..."},
            "spans": [
                {"name": "buscar_web", "tipo": "Span",
                 "input": {"query": "..."}, "output": {"n_resultados": 5}},
                {"name": "planificar", "tipo": "Generation",
                 "model": "gpt-4o-mini", "usage": {"input": 450, "output": 180}},
            ],
            "output": {"informe": "..."},
        }
    }
    import json
    print(json.dumps(estructura, indent=2, ensure_ascii=False))

## Cella 3 — `LangfuseTracer`: wrapper che avvolge il `Tracer` proprio

Invece di riscrivere `agent.py`, creiamo un wrapper che usa il `Tracer` proprio come fonte
di verità e invia i dati a Langfuse al termine. Se Langfuse non è disponibile,
degrada silenziosamente al comportamento locale.

In [ ]:
LF_TRACER_PY = '''
"""
LangfuseTracer: estende il Tracer proprio inviando dati a Langfuse.
Se Langfuse non è configurato, funziona come il Tracer locale.
"""
import os
from observability.tracer import Tracer, Span


class LangfuseTracer(Tracer):
    """
    Tracer con integrazione opzionale di Langfuse.
    Eredita tutto il comportamento del Tracer locale e aggiunge l'invio a Langfuse.

    Uso:
        tracer = LangfuseTracer(tema="...")  # Langfuse si attiva se ci sono credenziali
        # Il resto del codice è identico al Tracer locale
    """

    def __init__(self, tema: str, trazas_dir: str = "./trazas", tags: list[str] = None):
        super().__init__(tema=tema, trazas_dir=trazas_dir)
        self._lf_trace = None
        self._lf       = None
        self._tags     = tags or []
        self._init_langfuse(tema)

    def _init_langfuse(self, tema: str):
        """Inizializza Langfuse se le credenziali sono disponibili."""
        pk = os.getenv("LANGFUSE_PUBLIC_KEY")
        sk = os.getenv("LANGFUSE_SECRET_KEY")
        if not (pk and sk):
            return
        try:
            from langfuse import Langfuse
            self._lf = Langfuse(
                public_key=pk,
                secret_key=sk,
                host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            )
            self._lf_trace = self._lf.trace(
                id=self.trace_id,
                name="investigar",
                input={"tema": tema},
                tags=self._tags,
            )
        except Exception as e:
            # Non interrompe l'esecuzione se Langfuse non è disponibile
            self._lf = None
            self._lf_trace = None

    def registrar_generation(
        self,
        nombre: str,
        modelo: str,
        input_msgs: list[dict],
        output_txt: str,
        tokens_input: int,
        tokens_output: int,
        latencia_ms: float,
        prompt_name: str | None = None,
    ):
        """
        Registra una chiamata all'LLM come Generation in Langfuse
        (e la aggiunge anche al Tracer locale come Span).
        """
        import time as _time
        s = Span(nombre=nombre, modelo=modelo)
        s.inicio          = _time.perf_counter() - latencia_ms / 1000
        s.fin             = _time.perf_counter()
        s.tokens_input    = tokens_input
        s.tokens_output   = tokens_output
        s.ok              = True
        with self._lock:
            self.spans.append(s)

        if self._lf_trace:
            try:
                kwargs = dict(
                    name=nombre,
                    model=modelo,
                    input=input_msgs,
                    output=output_txt,
                    usage={"input": tokens_input, "output": tokens_output},
                    metadata={"latencia_ms": round(latencia_ms, 1)},
                )
                if prompt_name:
                    kwargs["prompt"] = self._lf.get_prompt(prompt_name)
                gen = self._lf_trace.generation(**kwargs)
                gen.end()
            except Exception:
                pass

    def puntuar(self, nombre: str, valor: float, comentario: str = ""):
        """Aggiunge un punteggio di qualità alla trace in Langfuse."""
        if self._lf_trace:
            try:
                self._lf_trace.score(
                    name=nombre,
                    value=valor,
                    comment=comentario,
                )
            except Exception:
                pass

    def cerrar(self, informe: str = "", coste_usd: float = 0.0):
        """Chiude la trace in Langfuse e salva la trace locale."""
        if self._lf_trace:
            try:
                self._lf_trace.update(
                    output={
                        "informe_preview": informe[:500] if informe else "",
                        "coste_usd": round(coste_usd, 6),
                        "tokens_totales": self.tokens_totales,
                        "duracion_s": round(self.duracion_total_s, 2),
                    }
                )
                self._lf.flush()
            except Exception:
                pass
        return self.guardar()
'''

OBS_DIR = PROJECT_DIR / "observability"
(OBS_DIR / "lf_tracer.py").write_text(LF_TRACER_PY.strip())
print("✓ observability/lf_tracer.py generato")

## Cella 4 — Gestione dei prompt con Langfuse

Migra il prompt di sintesi da `synthesizer.py` a Langfuse Prompt Management.
Il codice ha un fallback al prompt hardcoded se Langfuse non è disponibile.

In [ ]:
# Passo 1: crea il prompt in Langfuse (solo se sei connesso)
PROMPT_SINTESIS_V1 = """Sei un analista senior di scouting tecnologico.
A partire dai risultati di ricerca seguenti, genera un report
professionale e strutturato sul tema.

TEMA: {{tema}}

RISULTATI:
{{hallazgos}}

ISTRUZIONI:
- Scrivi in italiano, tono analitico e diretto
- Struttura: Riepilogo esecutivo → Risultati per area → Conclusioni → Fonti
- Sintetizza e interpreta, non copiare letteralmente
- Segnala tendenze chiave, attori rilevanti e gap informativi
- Formato Markdown pulito e leggibile"""

if LANGFUSE_ENABLED:
    try:
        # Crea il prompt in Langfuse se non esiste
        prompt_obj = langfuse.create_prompt(
            name="sintesis_informe",
            prompt=PROMPT_SINTESIS_V1,
            labels=["production"],   # segna come versione attiva
            config={"modelo": "gpt-4o-mini", "temperatura": 0.4},
        )
        print(f"✓ Prompt 'sintesis_informe' creato in Langfuse (versione {prompt_obj.version})")
        print(f"  Modificalo in: {LANGFUSE_HOST}/prompts")
    except Exception as e:
        print(f"⚠  Impossibile creare il prompt: {e}")
        print("   (potrebbe esistere già — controlla il pannello)")
else:
    print("[Modalità locale] Prompt che verrebbe caricato su Langfuse:")
    print(f"  Nome: sintesis_informe")
    print(f"  Variabili: {{{{tema}}}}, {{{{hallazgos}}}}")
    print(f"  Labels: [production]")

In [ ]:
# Passo 2: synthesizer.py aggiornato che recupera il prompt da Langfuse
SYNTHESIZER_V2_PY = '''
"""Sintetizzatore con prompt gestito da Langfuse (con fallback locale)."""
import os
from openai import OpenAI

MODELO_SINTESIS = "gpt-4o-mini"

# Prompt di fallback (usato se Langfuse non è disponibile)
_PROMPT_FALLBACK = """Sei un analista senior di scouting tecnologico.
A partire dai risultati seguenti, genera un report professionale su:

TEMA: {tema}

RISULTATI:
{hallazgos}

Scrivi in italiano. Struttura: Riepilogo esecutivo → Risultati → Conclusioni → Fonti.
Formato Markdown."""


def _obtener_prompt(tema: str, hallazgos: str) -> str:
    """Recupera il prompt da Langfuse oppure usa il fallback locale."""
    pk = os.getenv("LANGFUSE_PUBLIC_KEY")
    sk = os.getenv("LANGFUSE_SECRET_KEY")
    if pk and sk:
        try:
            from langfuse import Langfuse
            lf = Langfuse(
                public_key=pk, secret_key=sk,
                host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"),
            )
            # cache_ttl_seconds: non chiama Langfuse a ogni request
            p = lf.get_prompt("sintesis_informe", cache_ttl_seconds=300)
            return p.compile(tema=tema, hallazgos=hallazgos)
        except Exception:
            pass
    # Fallback: prompt hardcoded con .format()
    return _PROMPT_FALLBACK.format(tema=tema, hallazgos=hallazgos)


def generar_informe(tema: str, resultados: list[dict]) -> tuple[str, int, int]:
    """
    Genera il report finale.
    Restituisce (informe, tokens_input, tokens_output) per il tracer.
    """
    partes = []
    for r in resultados:
        fuentes_str = "\\n".join(f"  - {f}" for f in r.get("fuentes", []))
        partes.append(
            f"### Sottotask {r[\'subtarea_id\']}: {r[\'descripcion\']}\\n"
            f"{r[\'hallazgo\']}\\n"
            f"Fonti:\\n{fuentes_str or \'  (nessuna fonte esterna)\'}"
        )
    hallazgos_txt = "\\n\\n".join(partes)

    prompt_txt = _obtener_prompt(tema, hallazgos_txt)

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp   = client.chat.completions.create(
        model=MODELO_SINTESIS,
        messages=[{"role": "user", "content": prompt_txt}],
        temperature=0.4,
        max_tokens=2000,
    )
    informe = resp.choices[0].message.content
    t_in    = resp.usage.prompt_tokens
    t_out   = resp.usage.completion_tokens
    return informe, t_in, t_out
'''

(PROJECT_DIR / "agent" / "synthesizer.py").write_text(SYNTHESIZER_V2_PY.strip())
print("✓ agent/synthesizer.py aggiornato (recupera il prompt da Langfuse con fallback)")
print()
print("Cambio chiave:")
print("  generar_informe() ora restituisce (informe, tokens_input, tokens_output)")
print("  → token reali di OpenAI invece di stime")

## Cella 5 — `agent.py` integrato con `LangfuseTracer`

Aggiorna l'agente per usare `LangfuseTracer` e i token reali del sintetizzatore.

In [ ]:
AGENT_LF_PY = '''
"""ResearchAgent con Langfuse integrato."""
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Callable

from agent.planner     import crear_plan
from agent.executor    import ejecutar_subtarea
from agent.synthesizer import generar_informe
from observability.lf_tracer import LangfuseTracer
from observability.logger    import configurar_logger


@dataclass
class ResultadoInvestigacion:
    tema:         str
    plan:         list[dict]
    resultados:   list[dict]
    informe:      str
    duracion_s:   float
    n_fuentes:    int
    ruta_informe: str  = ""
    trace_id:     str  = ""
    coste_usd:    float = 0.0


class ResearchAgent:

    def __init__(
        self,
        informes_dir: str = "./informes",
        trazas_dir:   str = "./trazas",
        log_file:     str | None = None,
        verbose:      bool = True,
    ):
        self.informes_dir = Path(informes_dir)
        self.trazas_dir   = Path(trazas_dir)
        self.informes_dir.mkdir(parents=True, exist_ok=True)
        self.trazas_dir.mkdir(parents=True, exist_ok=True)
        self.verbose = verbose
        self.logger  = configurar_logger("research_agent", log_file=log_file)

    def _log(self, msg: str):
        if self.verbose:
            print(msg)

    def investigar(
        self,
        tema: str,
        on_progreso: Callable[[int, int, str], None] | None = None,
        tags: list[str] | None = None,
    ) -> ResultadoInvestigacion:

        tracer = LangfuseTracer(
            tema=tema,
            trazas_dir=str(self.trazas_dir),
            tags=tags or [],
        )
        self._log(f"\n🔍 [{tracer.trace_id}] {tema[:60]}\n")

        # ── Fase 1: Pianificazione ────────────────────────────────────────────
        self._log("📋 Fase 1/3 — Pianificazione...")
        with tracer.span("planificar") as s:
            plan       = crear_plan(tema)
            s.tokens_input  = 500
            s.tokens_output = len(plan) * 80
        self._log(f"   {len(plan)} sottotask")

        # ── Fase 2: Esecuzione ───────────────────────────────────────────────
        self._log("\n🔎 Fase 2/3 — Ricerca in corso...")
        resultados = []
        for i, subtarea in enumerate(plan, 1):
            with tracer.span(f"subtarea_{i}") as s:
                resultado = ejecutar_subtarea(subtarea)
                # Usa i token del risultato se executor li restituisce, altrimenti stima
                s.tokens_input  = resultado.get("tokens_input_total",
                                                len(subtarea["descripcion"]) * 4)
                s.tokens_output = resultado.get("tokens_output_total",
                                                len(resultado["hallazgo"]) // 4)
            resultados.append(resultado)
            if on_progreso:
                on_progreso(i, len(plan), subtarea["descripcion"])
            self._log(f"   {i}/{len(plan)} — {resultado[\'n_pasos\']} passi")

        # ── Fase 3: Sintesi ───────────────────────────────────────────────────
        self._log("\n✍️  Fase 3/3 — Sintesi...")
        with tracer.span("sintetizar") as s:
            # generar_informe ora restituisce token reali
            informe, t_in, t_out = generar_informe(tema, resultados)
            s.tokens_input  = t_in
            s.tokens_output = t_out

        # ── Salvataggio del report ────────────────────────────────────────────
        duracion_s = tracer.duracion_total_s
        n_fuentes  = sum(len(r["fuentes"]) for r in resultados)
        fecha      = datetime.now().strftime("%Y%m%d_%H%M")
        slug       = tema[:40].replace(" ", "_").replace("/", "-")
        ruta       = self.informes_dir / f"{fecha}_{slug}.md"

        cabecera = (
            f"---\ntema: {tema}\ntrace_id: {tracer.trace_id}\n"
            f"coste_usd: {tracer.coste_total_usd:.6f}\n"
            f"duracion_s: {duracion_s:.1f}\nn_fuentes: {n_fuentes}\n---\n\n"
        )
        ruta.write_text(cabecera + informe, encoding="utf-8")

        ruta_traza = tracer.cerrar(
            informe=informe,
            coste_usd=tracer.coste_total_usd,
        )

        self._log(
            f"\n✅ [{tracer.trace_id}]  {duracion_s:.1f}s  "
            f"${tracer.coste_total_usd:.5f}  {tracer.tokens_totales:,} tok"
        )

        return ResultadoInvestigacion(
            tema=tema, plan=plan, resultados=resultados,
            informe=informe, duracion_s=duracion_s, n_fuentes=n_fuentes,
            ruta_informe=str(ruta),
            trace_id=tracer.trace_id,
            coste_usd=tracer.coste_total_usd,
        )
'''

(PROJECT_DIR / "agent" / "agent.py").write_text(AGENT_LF_PY.strip())
print("✓ agent/agent.py aggiornato con LangfuseTracer")

## Cella 6 — Esecuzione con Langfuse attivo

In [ ]:
import importlib
import agent.agent, agent.synthesizer
importlib.reload(agent.synthesizer)
importlib.reload(agent.agent)

from agent.agent import ResearchAgent
from IPython.display import Markdown, display

agente = ResearchAgent(
    informes_dir=str(PROJECT_DIR / "informes"),
    trazas_dir=str(PROJECT_DIR / "trazas"),
    verbose=True,
)

resultado = agente.investigar(
    "Intelligenza artificiale nella filiera del vino galiziano",
    tags=["semana-18", "vino", "agrifood"],
)

print(f"\nTrace ID:  {resultado.trace_id}")
print(f"Costo:     ${resultado.coste_usd:.5f}")
print(f"Durata:    {resultado.duracion_s:.1f}s")

if LANGFUSE_ENABLED:
    print(f"\nVedi la trace: {LANGFUSE_HOST}/traces/{resultado.trace_id}")

In [ ]:
# Mostra il report
display(Markdown(resultado.informe))

## Cella 7 — Aggiungere un punteggio di qualità alla trace

Valuta il report con un LLM-as-judge (lo stesso pattern della settimana 8)
e collega il punteggio alla trace in Langfuse.

In [ ]:
import json
from openai import OpenAI

PROMPT_EVALUAR = """Valuta la qualità di questo report di scouting tecnologico.

REPORT:
{informe}

Assegna un punteggio da 0.0 a 1.0 per ogni dimensione e dai una breve motivazione.
Rispondi SOLO con JSON valido:
{{"relevancia": 0.0, "profundidad": 0.0, "estructura": 0.0,
  "media": 0.0, "comentario": "..."}}"""

def evaluar_informe(informe: str) -> dict:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user",
                   "content": PROMPT_EVALUAR.format(informe=informe[:3000])}],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    return json.loads(resp.choices[0].message.content)

print("Valutazione della qualità del report...")
puntuacion = evaluar_informe(resultado.informe)

print(f"\nPunteggi:")
for k, v in puntuacion.items():
    if k != "comentario":
        barra = "█" * int(v * 20) + "░" * (20 - int(v * 20))
        print(f"  {k:<15} {barra} {v:.2f}")
print(f"\nCommento: {puntuacion.get('comentario', '')}")

# Invia il punteggio a Langfuse se connesso
if LANGFUSE_ENABLED:
    trace = langfuse.trace(id=resultado.trace_id)
    for dim in ["relevancia", "profundidad", "estructura"]:
        if dim in puntuacion:
            trace.score(
                name=dim,
                value=float(puntuacion[dim]),
                comment=puntuacion.get("comentario", ""),
            )
    trace.score(name="calidad_media", value=float(puntuacion.get("media", 0)))
    langfuse.flush()
    print(f"\n✓ Punteggi inviati a Langfuse")
    print(f"  Vedi: {LANGFUSE_HOST}/traces/{resultado.trace_id}")

## Sfida — Confrontare due versioni del prompt di sintesi

Il vero potere del versionamento dei prompt è confrontare i risultati tra versioni.

**Crea una seconda versione del prompt** in Langfuse con queste differenze:
- Aggiungi l'istruzione: *"Includi sempre una sezione 'Implicazioni per il settore' alla fine"*
- Cambia il tono a: *"Scrivi per un dirigente non tecnico"*

Poi esegui l'agente due volte con lo stesso tema: una con la versione 1 (attiva) e una
con la versione 2, e confronta i report.

```python
# Per forzare una versione specifica di Langfuse:
p = langfuse.get_prompt("sintesis_informe", version=2)
prompt_txt = p.compile(tema=tema, hallazgos=hallazgos)
```

Valuta entrambi i report con `evaluar_informe()` e mostra il confronto:

| Dimensione | Versione 1 | Versione 2 |
|-----------|-----------|----------|
| relevancia | 0.82 | 0.79 |
| profundidad | 0.75 | 0.71 |
| estructura | 0.88 | 0.91 |

Quale versione è migliore secondo il giudice LLM? E secondo la tua lettura personale?

## Riepilogo della settimana 18

| Componente | File | Cosa offre |
|-----------|---------|------------|
| LangfuseTracer | `observability/lf_tracer.py` | Estende il Tracer locale con invio a Langfuse |
| Synthesizer v2 | `agent/synthesizer.py` | Recupera il prompt da Langfuse + token reali |
| Agent con tags | `agent/agent.py` | Tag per filtrare le trace nel pannello |
| Valutazione qualità | Cella 7 | LLM-as-judge collegato alla trace |
| Confronto prompt | Sfida | A/B test di versioni di prompt |

**Blocco 7 quasi completo.** La settimana 19 avvia il Blocco 8: strategia freelance —
come posizionarti, fissare i prezzi, ottenere il primo cliente e costruire il portfolio.